In [ ]:
import os
import sys

root_path = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.append(root_path)

In [ ]:
from hummingbot.strategy_v2.utils.distributions import Distributions
from controllers.market_making.pz_mm import PZMMControllerConfig
from core.backtesting.optimizer import BacktestingConfig, BaseStrategyConfigGenerator
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
from decimal import Decimal


class PZMMConfigGenerator(BaseStrategyConfigGenerator):
    """
    Strategy configuration generator for PZ MM optimization.
    """
    async def generate_config(self, trial) -> BacktestingConfig:

        # Those doesn't matter, they are dyncamically calculated inside the controller anyway
        take_profit = 5 # trial.suggest_float("take_profit", 0.01, 0.03, step=0.01)
        stop_loss = 5  #trial.suggest_float("stop_loss", 0.01, 0.05, step=0.01)


        # Controller configuration
        connector_name = "binance_perpetual"
        trading_pair = "WLD-USDT"
        total_amount_quote = 1000
 
        # levels = trial.suggest_int("levels", 3, 5)
        # start_spread = trial.suggest_float("start_spread", 0.002, 0.005, step=0.001)
        # step_spread = trial.suggest_float("step_spread", 0.001, 0.002, step=0.001)
        # spreads = Distributions.arithmetic(levels, start_spread, step_spread)
        # trailing_stop_activation_price = trial.suggest_float("trailing_stop_activation_price", 0.005, 0.015, step=0.01)
        # trailing_delta_ratio = trial.suggest_float("trailing_delta_ratio", 0.05, 0.1, step=0.01)
        # trailing_stop_trailing_delta = trailing_stop_activation_price * trailing_delta_ratio
        # time_limit = trial.suggest_int("time_limit", 30, 60 * 5, step=30)
        # executor_refresh_time = trial.suggest_int("executor_refresh_time", 30, 60, step=30)
        # cooldown_time = trial.suggest_int("cooldown_time", 60, 60 * 5, step=60)

        # tp_natr_factor = trial.suggest_float("tp_natr_factor", 0.25, 1, step=0.25)
        # sl_natr_factor = trial.suggest_float("sl_natr_factor", 0.5, 3, step=0.5)
        # hma_very_slow = trial.suggest_int("hma_very_slow", 30, 50, step = 5)
        # hma_slow = trial.suggest_int("hma_slow", 15, 30, step = 5)
        # hma_fast = trial.suggest_int("hma_fast", 5,15, step = 5)
        # rsi_length = trial.suggest_int("rsi_length", 5, 13, step = 2)
        # stoch_rsi_smoothing = trial.suggest_int("stoch_rsi_smoothing", 2, 6, step = 1)
        # stoch_rsi_length = trial.suggest_int("stoch_rsi_length", 5, 13, step = 2)
        # natr_length = trial.suggest_int("natr_length", 7, 21, step = 2)
        
        levels = 3 # trial.suggest_int("levels", 3, 5)
        start_spread = 0.002 # trial.suggest_float("start_spread", 0.002, 0.005, step=0.001)
        step_spread = 0.002 # trial.suggest_float("step_spread", 0.001, 0.002, step=0.001)
        spreads = Distributions.arithmetic(levels, start_spread, step_spread)

        trailing_stop_activation_price = trial.suggest_float("trailing_stop_activation_price", 0.005, 0.01, step=0.005)# trial.suggest_float("trailing_stop_activation_price", 0.005, 0.015, step=0.01)
        trailing_delta_ratio = 0.05 # trial.suggest_float("trailing_delta_ratio", 0.05, 0.1, step=0.01)
        trailing_stop_trailing_delta = trailing_stop_activation_price * trailing_delta_ratio
        time_limit = 60*60 #trial.suggest_int("time_limit", 30, 60 * 5, step=30)
        executor_refresh_time = 60 # trial.suggest_int("executor_refresh_time", 30, 60, step=30)
        cooldown_time = 60 #trial.suggest_int("cooldown_time", 60, 60 * 5, step=60)
        
        tp_natr_factor = 0.5 #trial.suggest_float("tp_natr_factor", 0.25, 1, step=0.25)
        sl_natr_factor = 3 # trial.suggest_float("sl_natr_factor", 0.5, 3, step=0.5)
        hma_very_slow = 50 # trial.suggest_int("hma_very_slow", 30, 50, step = 5)
        hma_slow = 30 #trial.suggest_int("hma_slow", 15, 30, step = 5)
        hma_fast = 10 #trial.suggest_int("hma_fast", 5,15, step = 5)
        rsi_length = 9 #trial.suggest_int("rsi_length", 5, 13, step = 2)
        stoch_rsi_smoothing = 3 #  trial.suggest_int("stoch_rsi_smoothing", 2, 6, step = 1)
        stoch_rsi_length = 9 #trial.suggest_int("stoch_rsi_length", 5, 13, step = 2)
        natr_length = 14 # trial.suggest_int("natr_length", 7, 21, step = 2)




        # Creating the instance of the configuration and the controller
        config = PZMMControllerConfig(
            connector_name=connector_name,
            trading_pair=trading_pair,
            sell_spreads=spreads,
            buy_spreads=spreads,
            total_amount_quote=Decimal(total_amount_quote),
            take_profit=Decimal(take_profit),
            stop_loss=Decimal(stop_loss),
            trailing_stop=TrailingStop(activation_price=Decimal(trailing_stop_activation_price), trailing_delta=Decimal(trailing_stop_trailing_delta)),
            time_limit=time_limit,
            cooldown_time=cooldown_time,
            executor_refresh_time=executor_refresh_time,
            hma_very_slow = hma_very_slow,
            hma_slow = hma_slow,
            hma_fast = hma_fast,
            rsi_length = rsi_length,
            stoch_rsi_smoothing = stoch_rsi_smoothing,
            stoch_rsi_length =stoch_rsi_length,
            natr_length = natr_length,
            sl_natr_factor=sl_natr_factor,
            tp_natr_factor=tp_natr_factor,
        )

        # Return the configuration encapsulated in BacktestingConfig
        return BacktestingConfig(config=config, start=self.start, end=self.end)

In [ ]:
from core.backtesting.optimizer import StrategyOptimizer
optimizer = StrategyOptimizer(root_path=root_path)

In [ ]:
optimizer.launch_optuna_dashboard()

In [ ]:
import datetime

start_date = datetime.datetime(2025, 3, 30)
end_date = datetime.datetime(2025, 3, 31)
# start_date = datetime.datetime(2024, 8, 2)
# end_date = datetime.datetime(2024, 8, 3)
config_generator = PZMMConfigGenerator(start_date=start_date, end_date=end_date)

await optimizer.optimize(
    study_name="pz_mm_with_optimizer",
    config_generator=config_generator,
    n_trials=100,
)

In [ ]:
optimizer.kill_optuna_dashboard()